<a href="https://colab.research.google.com/github/bruno-ritter/brazilain_stock_volatility_and_google_trends/blob/main/POC_TCC_Google_Trends_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tarefas

*  Limpar a base antes de rodar os modelos: retirar nan, retirar svi não-estacionários
*  Fazer o modelo multivariado GARCH
* Comparar os três modelos









# Bibliotecas

In [2]:
!pip install yfinance pandas pytrends tqdm arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 14.1 MB/s eta 0:00:00


In [3]:
import time
import math
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import arch

from pytrends.request import TrendReq
from datetime import datetime, timedelta

from arch import arch_model
from arch.univariate import ConstantMean, GARCH, Normal


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Bases salvas

In [5]:
df_acoes = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/acoes_elegiveis_v3.csv")
df_trends = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/google_trends_svi.csv", parse_dates=["semana"])
df_precos = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/precos_semanais_v3.csv", parse_dates=["semana"])
df_log = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_coleta_svi.csv")
df_adf = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_adf_svi.csv")
df_final = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/base_final_tcc.csv", parse_dates=["semana"])

# Coleta dos dados

Para cada ação, coleta o preço no yahoo finance e o SVI com o ticker como termo de busca:
  1. TICKER     — ex: "PETR4"

In [ ]:
# ──────────────────────────────────────────────────────────────
# CONFIGURAÇÕES
# ──────────────────────────────────────────────────────────────
ANOS_HISTORICO    = 5
VOLUME_MIN_DIARIO = 1
SEMANAS           = 260
DATA_FIM          = datetime(2026, 3, 15).strftime("%Y-%m-%d")
DATA_INICIO       = (datetime(2026, 3, 15) - timedelta(weeks=SEMANAS)).strftime("%Y-%m-%d")
PAUSA_TRENDS      = 12     # segundos entre requisições
CAMINHO  = "/content/drive/MyDrive/TCC - Youtube e mercado financeiro/"

## Dados financeiros

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 1 — PREÇOS
# ──────────────────────────────────────────────────────────────

# Todos os ticker
tickers = pd.read_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/acoes-listadas-b3.csv")
TERMOS = list(tickers.Ticker)
tickers_yf = [t + ".SA" for t in TERMOS]

dados_raw = yf.download(
    tickers_yf, start=DATA_INICIO, end=DATA_FIM,
    auto_adjust=True, progress=True,
)

registros_acoes = []
for ticker_yf in tickers_yf:

    try:
        close  = dados_raw["Close"][ticker_yf].dropna()
        volume = dados_raw["Volume"][ticker_yf].dropna()

        if len(close) < ANOS_HISTORICO * 252 * 0.85:
            print(f"  [SKIP] {ticker_yf} — histórico insuficiente ({len(close)} dias)")
            continue

        vol_fin = (volume * close.mean()).mean()
        if vol_fin < VOLUME_MIN_DIARIO:
            print(f"  [SKIP] {ticker_yf} — liquidez insuficiente")
            continue

        registros_acoes.append({
            "ticker_yf": ticker_yf,
            "dias": len(close),
            "vol_fin_medio": round(vol_fin, 2),
        })
        print(f"  [OK] {ticker_yf} | Vol R${vol_fin:>14,.0f}")

    except Exception as e:
        print(f"  [ERRO] {ticker_yf}: {e}")

df_acoes = pd.DataFrame(registros_acoes)
df_acoes.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/acoes_elegiveis_v3.csv", index=False)
print(f"\n✅ {len(df_acoes)} ações elegíveis → acoes_elegiveis_v3.csv")

[**********************97%********************** ]  375 of 387 completedERROR:yfinance:Failed to get ticker 'ALPK3.SA' reason: Failed to perform, curl: (28) Operation timed out after 10000 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  387 of 387 completed
ERROR:yfinance:
4 Failed downloads:
ERROR:yfinance:['AZEV11.SA', 'BIOM11.SA', 'PINE11.SA', 'ALPK3.SA']: YFTzMissingError('possibly delisted; no timezone found')


  [OK] PETR4.SA | Vol R$ 1,210,975,118
  [OK] B3SA3.SA | Vol R$   501,119,526
  [OK] HAPV3.SA | Vol R$   409,380,887
  [OK] BEEF3.SA | Vol R$    81,652,350
  [OK] GMAT3.SA | Vol R$    34,957,446
  [OK] ENEV3.SA | Vol R$   122,175,356
  [OK] CSAN3.SA | Vol R$   201,455,924
  [OK] RAIZ4.SA | Vol R$    61,929,945
  [OK] COGN3.SA | Vol R$    87,962,749
  [OK] MBRF3.SA | Vol R$   114,409,379
  [OK] BBDC4.SA | Vol R$   560,890,488
  [OK] VAMO3.SA | Vol R$    63,007,929
  [OK] ITUB4.SA | Vol R$   772,043,903
  [OK] ABEV3.SA | Vol R$   314,727,503
  [OK] CPLE3.SA | Vol R$    34,169,765
  [OK] BBAS3.SA | Vol R$   490,705,722
  [OK] ITSA4.SA | Vol R$   207,866,784
  [OK] CSNA3.SA | Vol R$   137,580,532
  [OK] VALE3.SA | Vol R$ 1,486,093,158
  [OK] PRIO3.SA | Vol R$   466,796,500
  [OK] MGLU3.SA | Vol R$   723,354,399
  [OK] PETR3.SA | Vol R$   343,194,967
  [OK] CMIG4.SA | Vol R$    97,536,101
  [OK] ONCO3.SA | Vol R$    23,598,592
  [OK] CVCB3.SA | Vol R$   108,319,470
  [SKIP] NATU3.SA — histó

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 2 — VOLATILIDADE REALIZADA
# ──────────────────────────────────────────────────────────────

series_semanais = []
for _, row in df_acoes.iterrows():
    try:
        close  = dados_raw["Close"][row["ticker_yf"]].dropna()
        volume = dados_raw["Volume"][row["ticker_yf"]].dropna()

        df_d = pd.DataFrame({"close": close, "volume": volume})
        df_d.index = pd.to_datetime(df_d.index)

        df_d["log_ret"] = np.log(df_d["close"] / df_d["close"].shift(1))

        # Agrega semanalmente (semana fecha na sexta)
        wk = df_d.resample("W-FRI").agg(
            preco_abertura   = ("close",   "first"),
            preco_fechamento = ("close",   "last"),
            volume_total     = ("volume",  "sum"),
            n_preços         = ("log_ret", "count"),      # nº de pregões na semana
            rv               = ("log_ret", lambda x: (x**2).sum()),  # RV = Σ r²
        ).dropna(subset=["rv"])

        wk["ticker_yf"] = row["ticker_yf"]
        wk["ticker_b3"] = row["ticker_yf"].replace(".SA", "")
        series_semanais.append(wk.reset_index().rename(columns={"Date": "semana"}))

    except Exception as e:
        print(f"  [ERRO preços {row['ticker_yf']}]: {e}")

df_precos = pd.concat(series_semanais, ignore_index=True)

# Retirar ações que não possuem o número de semanas correto
ticker_counts = df_precos.groupby("ticker_b3")["semana"].count()
tickers_corretos = ticker_counts[ticker_counts == SEMANAS].index
df_precos = df_precos[df_precos["ticker_b3"].isin(tickers_corretos)]


df_precos.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/precos_semanais_v3.csv", index=False)
print(f"✅ {len(df_precos)} linhas → precos_semanais_v3.csv")

✅ 86320 linhas → precos_semanais_v3.csv


## GOOGLE TRENDS: SVI POR TICKER

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 3 — GOOGLE TRENDS: SVI POR TICKER
# ──────────────────────────────────────────────────────────────

pytrends = TrendReq(hl="pt-BR", tz=-180)

def coletar_svi(ticker):
    """Retorna DataFrame semanal do SVI para um ticker ou None."""
    try:
        pytrends.build_payload(
            kw_list=[ticker],
            timeframe=f"{DATA_INICIO} {DATA_FIM}",
            geo="BR",
            gprop="",
        )
        df = pytrends.interest_over_time()
        if df.empty:
            return None
        df = df.drop(columns=["isPartial"], errors="ignore")
        df.columns = ["svi"]
        df.index.name = "semana"
        return df.reset_index()
    except Exception as e:
        print(f"    [ERRO '{ticker}']: {e}")
        return None


todos_trends = []
log_coleta   = []
tickers_validos = list(df_precos["ticker_b3"].unique())

for ticker in tickers_validos:

    print(f"\n  {ticker}")
    df_t = coletar_svi(ticker)
    time.sleep(PAUSA_TRENDS)

    if df_t is None:
        print("VAZIO")
        log_coleta.append({
            "ticker": ticker, "status": "vazio",
            "semanas": 0, "svi_media": None, "svi_std": None,
        })
        continue

    svi_media = round(df_t["svi"].mean(), 2)
    svi_std   = round(df_t["svi"].std(), 2)
    svi_zeros = (df_t["svi"] == 0).mean()
    print(f"OK | {len(df_t)} sem | média={svi_media:.1f} "
          f"std={svi_std:.1f} zeros={svi_zeros:.0%}")

    for _, tr in df_t.iterrows():
        todos_trends.append({
            "semana": tr["semana"],
            "ticker_b3": ticker,
            "svi":    tr["svi"],
        })

    log_coleta.append({
        "ticker":       ticker,
        "status":       "ok",
        "semanas":      len(df_t),
        "svi_media":    svi_media,
        "svi_std":      svi_std,
        "svi_zeros_pct": round(svi_zeros * 100, 1),
    })


df_trends = pd.DataFrame(todos_trends)
df_log    = pd.DataFrame(log_coleta)

df_trends.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/google_trends_svi.csv", index=False)
df_log.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_coleta_svi.csv", index=False)

print(f"\n✅ {len(df_trends)} linhas → google_trends_svi.csv")
print(f"✅ Log de coleta    → log_coleta_svi.csv")


  PETR4
OK | 261 sem | média=30.6 std=10.2 zeros=0%

  B3SA3
OK | 261 sem | média=19.6 std=10.6 zeros=0%

  HAPV3
OK | 261 sem | média=10.7 std=9.1 zeros=0%

  BEEF3
OK | 261 sem | média=26.2 std=12.3 zeros=0%

  GMAT3
OK | 261 sem | média=26.0 std=15.2 zeros=1%

  ENEV3
OK | 261 sem | média=14.6 std=11.7 zeros=3%

  CSAN3
OK | 261 sem | média=19.0 std=9.5 zeros=0%

  COGN3
OK | 261 sem | média=34.5 std=15.7 zeros=0%

  MBRF3
OK | 261 sem | média=4.4 std=14.6 zeros=90%

  BBDC4
OK | 261 sem | média=24.1 std=9.2 zeros=0%

  VAMO3
OK | 261 sem | média=30.5 std=20.4 zeros=0%

  ITUB4
OK | 261 sem | média=40.7 std=11.2 zeros=0%

  ABEV3
OK | 261 sem | média=11.8 std=7.2 zeros=0%

  CPLE3
OK | 261 sem | média=39.4 std=18.9 zeros=0%

  BBAS3
OK | 261 sem | média=19.8 std=13.0 zeros=0%

  ITSA4
OK | 261 sem | média=41.9 std=13.3 zeros=0%

  CSNA3
OK | 261 sem | média=46.9 std=13.6 zeros=0%

  VALE3
OK | 261 sem | média=39.7 std=10.5 zeros=0%

  PRIO3
OK | 261 sem | média=36.6 std=10.2 zeros=

## ESTACIONARIEDADE DO SVI (ADF POR TICKER)

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 4 — ESTACIONARIEDADE DO SVI (ADF POR TICKER)
# ──────────────────────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

NIVEL_SIGNIFICANCIA = 0.05

def teste_adf(serie, nome=""):
    """
    Roda o ADF e retorna dict com estatística, p-valor e conclusão.
    Usa seleção automática de lags (método 'AIC').
    """
    serie = serie.dropna()
    if len(serie) < 10:
        return {"stat": None, "pvalor": None, "estacionaria": None, "obs": "série curta"}

    stat, pvalor, _, _, valores_criticos, _ = adfuller(serie, autolag="AIC")
    estacionaria = pvalor < NIVEL_SIGNIFICANCIA

    # print(f"      {'[OK]' if estacionaria else '[RAIZ UNITÁRIA]'} {nome}: "
          # f"stat={stat:.3f}  p={pvalor:.4f}  {'estacionária' if estacionaria else 'não estacionária'}")

    return {
        "stat":         round(stat, 4),
        "pvalor":       round(pvalor, 4),
        "cv_1pct":      round(valores_criticos["1%"], 3),
        "cv_5pct":      round(valores_criticos["5%"], 3),
        "cv_10pct":     round(valores_criticos["10%"], 3),
        "estacionaria": estacionaria,
        "obs":          "",
    }


resultados_adf = []

for ticker, grupo in df_trends.groupby("ticker"):

    # print(f"\n  {ticker}")
    svi = grupo.sort_values("semana")["svi"].reset_index(drop=True)

    # ── ADF no nível ──────────────────────────────────────────
    adf_nivel = teste_adf(svi, "SVI nível")

    # ── Primeira diferença e ADF ──────────────────────────────
    svi_diff  = svi.diff()
    adf_diff  = teste_adf(svi_diff, "SVI Δ(1)")

    resultados_adf.append({
        "ticker":                   ticker,
        # nível
        "svi_adf_stat":             adf_nivel["stat"],
        "svi_adf_pvalor":           adf_nivel["pvalor"],
        "svi_estacionario":         adf_nivel["estacionaria"],
        # primeira diferença
        "svi_diff_adf_stat":        adf_diff["stat"],
        "svi_diff_adf_pvalor":      adf_diff["pvalor"],
        "svi_diff_estacionario":    adf_diff["estacionaria"],
    })

    # ── Adiciona coluna svi_diff ao df_trends ─────────────────
    df_trends.loc[df_trends["ticker"] == ticker, "svi_diff"] = svi_diff.values


# ── Bases finais ──────────────────────────────────────────────
df_adf = pd.DataFrame(resultados_adf)

df_adf.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/log_adf_svi.csv", index=False)

# ── Resumo ────────────────────────────────────────────────────
n_nivel_ok = df_adf["svi_estacionario"].sum()
n_diff_ok  = df_adf["svi_diff_estacionario"].sum()
n          = len(df_adf)

print(f"\n{'─'*55}")
print(f"  SVI nível estacionário:      {n_nivel_ok}/{n} tickers")
print(f"  SVI Δ(1)  estacionário:      {n_diff_ok}/{n} tickers")
print(f"\n✅ google_trends_svi.csv  → svi + svi_diff por semana")
print(f"✅ log_adf_svi.csv        → resultado ADF por ticker")

KeyError: 'ticker'

## Base de dados final

In [ ]:
# ──────────────────────────────────────────────────────────────
# ETAPA 5 — MERGE: PREÇOS + TRENDS
# Uma linha por (semana, ticker)
# ──────────────────────────────────────────────────────────────

# Ensure 'semana' is datetime (redundant if KEXjcGZ7Sk8a is run, but good for robustness)
df_trends["semana"] = pd.to_datetime(df_trends["semana"])
df_precos["semana"] = pd.to_datetime(df_precos["semana"])

# Add ticker_b3 to df_trends BEFORE selecting columns for merge
df_trends["ticker_b3"] = df_trends["ticker"]

# Alinha domingo (Trends) → sexta (preços)
df_trends["semana"] = df_trends["semana"] + pd.offsets.Week(weekday=4)

df_final = pd.merge(
    df_precos,
    df_trends[["semana","ticker_b3","svi","svi_diff"]],
    on=["semana","ticker_b3"],
    how="inner",
).sort_values(["ticker_b3","semana"]).reset_index(drop=True)

# Selecionar apenas o ticker com SVI Diff estacionário
df_final = df_final[df_final["ticker_b3"].isin(df_adf[df_adf['svi_diff_estacionario'] == True]["ticker"])]

# Retirar os valores NaN da primeira diferença do svi
df_final = df_final[df_final["svi_diff"].notna()]

# Retirar ativos com número de semanas errada
df_final["semana"] = pd.to_datetime(df_final["semana"])
ticker_counts = df_final.groupby("ticker_b3")["semana"].count()
tickers_retirar = ticker_counts[ticker_counts != SEMANAS].index
df_final = df_final[df_final["ticker_b3"].isin(tickers_retirar)]

df_final.to_csv("/content/drive/MyDrive/TCC - Youtube e mercado financeiro/base_final_tcc.csv", index=False)

print(f"✅ BASE FINAL → base_final_tcc_v3.csv")

# Especificação dos Modelos Competidores

## MODELO 0 — PASSEIO ALEATÓRIO (Random Walk)
σ̂²_t = RV_{t-1}

In [9]:
def passeio_aleatorio(grupo):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv"]]
        .copy()
    )
    df_t["rv_pred_m0"] = df_t["rv"].shift(1)
    return df_t.dropna(subset=["rv_pred_m0"])

## MODELO 1: GARCH(1,1) PADRÃO
σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}

Janela móvel de tamanho fixo: 52 * 3 semanas

In [10]:
def garch_padrao(grupo, janela=52 * 3):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv"]]
        .copy()
    )
    n, previsoes = len(df_t), []

    for t in range(janela, n):
        treino = df_t["rv"].iloc[t - janela : t].values.astype(float)
        try:
            fit = arch_model(treino, mean="Constant", vol="GARCH",
                             p=1, q=1, dist="normal").fit(disp="off", show_warning=False)
            var_pred = fit.forecast(horizon=1, reindex=False).variance.values[-1, 0] / 10_000
            previsoes.append({
                "semana":     df_t["semana"].iloc[t],
                "rv":         df_t["rv"].iloc[t],
                "rv_pred_m1": var_pred,
                "omega":      fit.params.get("omega",    np.nan),
                "alpha":      fit.params.get("alpha[1]", np.nan),
                "beta":       fit.params.get("beta[1]",  np.nan),
            })
        except Exception as e:
            print(f"  [ERRO GARCH t={t}]: {e}")
            previsoes.append({"semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t],
                              "rv_pred_m1": np.nan, "omega": np.nan, "alpha": np.nan, "beta": np.nan})

    return pd.DataFrame(previsoes)

## MODELO 2: GARCH-X(1,1)
$(σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1} + γ·ΔSVI_{t-1})$

Janela móvel de tamanho fixo: 52 * 3 semanas

---
O que muda em relação ao Modelo 1:

A única diferença estrutural é o exógeno $\gamma \cdot \Delta SVI_{t-1}$, que aparece em dois momentos distintos:

**Na estimação** — `x=svi_treino` passa os 156 valores de $\Delta SVI$ da janela para a biblioteca `arch` estimar o parâmetro $\gamma$ junto com $(\omega, \alpha, \beta)$.

**Na previsão** — `x=svi_previsao` passa apenas o valor $\Delta SVI_{t-1}$, que é o único ponto do exógeno disponível no momento em que a previsão é feita. Isso garante que não há *look-ahead bias* — na semana $t$ você só conhece o SVI até a semana $t-1$.
```
Janela de treino:   svi_diff[ t-156 … t-1 ]  →  estima γ
Previsão:           svi_diff[ t-1 ]           →  usa γ para prever σ²_t


In [11]:
def garch_exog(grupo, col_exog="svi_diff", janela=52 * 3):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", col_exog]]
        .dropna(subset=[col_exog])
        .reset_index(drop=True)
        .copy()
    )
    n, previsoes = len(df_t), []

    if n <= janela:
        print(f"  [AVISO] Obs. insuficientes ({n}), retornando vazio.")
        return pd.DataFrame()

    for t in range(janela, n):
        treino        = df_t["rv"].iloc[t - janela : t].values.astype(float)
        exog_treino   = df_t[col_exog].iloc[t - janela : t].values.astype(float).reshape(-1, 1)
        exog_forecast = df_t[col_exog].iloc[t : t + 1].values.astype(float).reshape(1, 1)
        try:
            fit = arch_model(treino, x=exog_treino, mean="ARX", vol="GARCH",
                             p=1, q=1, dist="normal").fit(disp="off", show_warning=False)
            var_pred = fit.forecast(horizon=1, reindex=False, x=exog_forecast).variance.values[-1, 0] / 10_000

            gamma_key  = next((k for k in fit.params.index  if k.startswith("x")), None)
            gamma      = fit.params[gamma_key]  if gamma_key else np.nan
            gamma_pval = fit.pvalues[gamma_key] if gamma_key else np.nan

            previsoes.append({
                "semana":      df_t["semana"].iloc[t],
                "rv":          df_t["rv"].iloc[t],
                "rv_pred_m2":  var_pred,
                "omega":       fit.params.get("omega",    np.nan),
                "alpha":       fit.params.get("alpha[1]", np.nan),
                "beta":        fit.params.get("beta[1]",  np.nan),
                "gamma":       gamma,
                "gamma_pval":  gamma_pval,
            })
        except Exception as e:
            print(f"  [ERRO GARCH-X t={t}]: {e}")
            previsoes.append({"semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t],
                              "rv_pred_m2": np.nan, "omega": np.nan, "alpha": np.nan,
                              "beta": np.nan, "gamma": np.nan, "gamma_pval": np.nan})

    return pd.DataFrame(previsoes)

# Avaliação de Previsão e Inferência (Modelos Aninhados)



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Configurações ─────────────────────────────────────────────────
JANELA   = 52 * 3

# ── Loop principal ────────────────────────────────────────────────
resultados, resumo_params = [], []

for ticker, grupo in df_final.groupby("ticker_b3"):
    print(f"► {ticker}", end=" ... ")

    df_t = grupo.sort_values("semana").reset_index(drop=True)
    if len(df_t) <= JANELA:
        print("insuficiente, pulando.")
        continue

    semana_corte = df_t["semana"].iloc[JANELA]

    df_m0 = passeio_aleatorio(grupo)
    df_m1 = garch_padrao(grupo, janela=JANELA)
    df_m2 = garch_exog(grupo, janela=JANELA)

    if df_m2.empty:
        continue

    # ── Merge ────────────────────────────────────────────────────
    df_tick = (
        df_m1
        .merge(df_m0[["semana", "rv_pred_m0"]], on="semana", how="left")
        .merge(df_m2[["semana", "rv_pred_m2", "gamma", "gamma_pval"]], on="semana", how="left")
    )
    df_tick["ticker_b3"] = ticker
    df_tick["split"]     = np.where(df_tick["semana"] >= semana_corte,
                                    "out-of-sample", "in-sample")
    resultados.append(df_tick)

    # ── Resumo parâmetros M2 (todas as janelas são OOS) ──────────
    resumo_params.append({
        "ticker_b3":        ticker,
        "n_previsoes":      len(df_m2),
        "gamma_medio":      df_m2["gamma"].mean(),
        "gamma_pval_medio": df_m2["gamma_pval"].mean(),
        "pct_signif_5pct":  (df_m2["gamma_pval"] < 0.05).mean() * 100,
        "persistencia_m1":  (df_m1["alpha"] + df_m1["beta"]).mean(),
        "persistencia_m2":  (df_m2["alpha"] + df_m2["beta"]).mean(),
    })
    print("✓")

# ── Consolida ─────────────────────────────────────────────────────
df_previsoes     = pd.concat(resultados, ignore_index=True)
df_resumo_params = pd.DataFrame(resumo_params).set_index("ticker_b3")

# ── QLIKE (apenas OOS) ───────────────────────────────────────────
def qlike(rv, pred):
    mask = (rv > 0) & (pred > 0)
    r    = np.where(mask, rv / pred, np.nan)
    return np.where(mask, r - np.log(r) - 1, np.nan)

df_oos = df_previsoes.query("split == 'out-of-sample'").copy()
df_oos["qlike_m0"] = qlike(df_oos["rv"], df_oos["rv_pred_m0"])
df_oos["qlike_m1"] = qlike(df_oos["rv"], df_oos["rv_pred_m1"])
df_oos["qlike_m2"] = qlike(df_oos["rv"], df_oos["rv_pred_m2"])

df_qlike = (
    df_oos.groupby("ticker_b3")[["qlike_m0", "qlike_m1", "qlike_m2"]]
    .mean()
    .rename(columns={"qlike_m0": "M0 (RW)", "qlike_m1": "M1 (GARCH)", "qlike_m2": "M2 (GARCH-X)"})
)
df_qlike["melhor"] = df_qlike.idxmin(axis=1)

# ── Salva ─────────────────────────────────────────────────────────
df_previsoes.to_csv(CAMINHO + "previsoes_consolidadas.csv", index=False)
df_resumo_params.to_csv(CAMINHO + "resumo_parametros.csv")

print(f"\n✅ {len(df_previsoes)} linhas | "
      f"{(df_previsoes['split']=='out-of-sample').sum()} OOS")

# ════════════════════════════════════════════════════════════════
# VISUALIZAÇÃO
# ════════════════════════════════════════════════════════════════

# ── Resumo textual ────────────────────────────────────────────────
n_sig = (df_resumo_params["pct_signif_5pct"] >= 50).sum()
print(f"\n📊 Resumo γ (ΔSVI):")
print(f"   {n_sig}/{len(df_resumo_params)} tickers com γ significativo em ≥50% das janelas OOS")
print(f"\n   Top 5 por % de janelas significativas:")
print(df_resumo_params["pct_signif_5pct"].sort_values(ascending=False).head().to_string())
print(f"\n📊 Modelo vencedor (QLIKE):")
print(df_qlike["melhor"].value_counts().to_string())

► AALR3 ... ✓
► ABCB4 ... ✓
► ABEV3 ... ✓
► AERI3 ... ✓
► AGRO3 ... ✓
► AGXY3 ... ✓
► AHEB3 ... ✓
► ALLD3 ... ✓
► ALOS3 ... ✓
► ALPA3 ... ✓
► ALPA4 ... ✓
► ALPK3 ... ✓
► ALUP11 ... ✓
► ALUP3 ... ✓
► ALUP4 ... ✓
► AMAR3 ... ✓
► AMBP3 ... ✓
► AMER3 ... ✓
► ANIM3 ... ✓
► ARML3 ... ✓
► ASAI3 ... ✓
► AVLL3 ... ✓
► AXIA3 ...   [ERRO GARCH-X t=156]: Singular matrix
  [ERRO GARCH-X t=157]: Singular matrix
  [ERRO GARCH-X t=158]: Singular matrix
  [ERRO GARCH-X t=159]: Singular matrix
  [ERRO GARCH-X t=160]: Singular matrix
  [ERRO GARCH-X t=161]: Singular matrix
  [ERRO GARCH-X t=162]: Singular matrix
  [ERRO GARCH-X t=163]: Singular matrix
  [ERRO GARCH-X t=164]: Singular matrix
  [ERRO GARCH-X t=165]: Singular matrix
  [ERRO GARCH-X t=166]: Singular matrix
  [ERRO GARCH-X t=167]: Singular matrix
  [ERRO GARCH-X t=168]: Singular matrix
  [ERRO GARCH-X t=169]: Singular matrix
  [ERRO GARCH-X t=170]: Singular matrix
  [ERRO GARCH-X t=171]: Singular matrix
  [ERRO GARCH-X t=172]: Singular matrix